In [ ]:
import numpy as np
import pandas as pd

### Load datasets for modeling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

premodel_core = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Core_Final.csv")
premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")

Mounted at /content/drive


/tmp/ipython-input-841/274579847.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")


### Quick dataset checks

In [ ]:
print("Core Shape:", premodel_core.shape)
print("Gust Shape:", premodel_gust.shape)
print("Core Class Balance Check:", premodel_core["High_Intensity"].value_counts(normalize=True))
print("Gust Class Balance Check:", premodel_gust["High_Intensity"].value_counts(normalize=True))

Core Shape: (1563080, 21)
Gust Shape: (1011445, 25)
Core Class Balance Check: High_Intensity
0    0.900896
1    0.099104
Name: proportion, dtype: float64
Gust Class Balance Check: High_Intensity
0    0.886772
1    0.113228
Name: proportion, dtype: float64


In [ ]:
premodel_core["LC_Type1"] = premodel_core["LC_Type1"].astype("category")
premodel_gust["LC_Type1"] = premodel_gust["LC_Type1"].astype("category")

print(premodel_core.dtypes)
print(premodel_gust.dtypes)


High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
LC_Type1                        category
Month                              int64
Day_of_Year                        int64
Latitude_Fire                    float64
Longitude_Fire                   float64
Elevation_m                      float64
FRP_max                          float64
Fire_ID                            int64
Climate_ID                        object
LC_Label                          object
Nearest_Core_Station_Dist_km     float64
Core_Dist_75km                      bool
Core_Dist_100km                     bool
Confidence_Ordered                 int64
DayNight                          object
Hour                               int64
Year                               int64
Province                          object
dtype: object
High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
Ma

# **LR/RF ENSEMBLE FOR CORE DATASET**

## **(1) Define Features and Response**

In [ ]:
# Core feature set for baseline model
core_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year",
    "LC_Type1"
]

X = premodel_core[core_features].copy()
y = premodel_core["High_Intensity"].copy()

### **(2) Define Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Startified split to ensure rare high-intensity class representation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

### **(3) Feature Adjustment Preprocessing**

In [ ]:
# Define feature types (numerical and categorical)
numeric_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year"
]

categorical_features = ["LC_Type1"]

### Construct preprocessing and model pipeline for LOGISTIC REGRESSION

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=123
)

LR_pipeline = Pipeline(
    steps=[("preprocess", preprocess), ("model", log_reg)]
)

### Construct preprocessing and model pipeline for RANDOM FOREST

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# No scaling for numeric, only encoding for land cover
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Input random forest model parameters
RF = RandomForestClassifier(
    n_estimators=300,
    random_state=123,
    n_jobs=-1,
    class_weight="balanced_subsample",
    min_samples_leaf=5
)

# Define pipeline
RF_pipeline = Pipeline(
    steps=[("preprocess", preprocess), ("model", RF)]
)

### **(4) Fit Model & Evaluate Performance**

In [ ]:
# Fit the model for LOGISTIC REGRESSION
LR_pipeline.fit(X_train, y_train)

# Evaluate Performance
y_pred = LR_pipeline.predict(X_test)
LR_prob = LR_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC–AUC:", roc_auc_score(y_test, LR_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.93      0.51      0.65    281634
           1       0.12      0.63      0.21     30982

    accuracy                           0.52    312616
   macro avg       0.53      0.57      0.43    312616
weighted avg       0.85      0.52      0.61    312616

ROC–AUC: 0.5961600677339053
Confusion Matrix:
 [[142478 139156]
 [ 11325  19657]]


In [ ]:
# Fit the model for RANDOM FOREST
RF_pipeline.fit(X_train, y_train)

# Evaluate Performance
y_pred = RF_pipeline.predict(X_test)
RF_prob = RF_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC–AUC:", roc_auc_score(y_test, RF_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.90      0.93    281634
           1       0.44      0.70      0.54     30982

    accuracy                           0.88    312616
   macro avg       0.70      0.80      0.74    312616
weighted avg       0.91      0.88      0.89    312616

ROC–AUC: 0.9035124416010074
Confusion Matrix:
 [[253839  27795]
 [  9165  21817]]


### Calculate ensemble probabilities/class predictions and resulting performance metrics

In [ ]:
ensemble_prob = 0.7 * RF_prob + 0.3 * LR_prob
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

print(classification_report(y_test, ensemble_pred))
print("ROC–AUC:", roc_auc_score(y_test, ensemble_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, ensemble_pred))

              precision    recall  f1-score   support

           0       0.97      0.90      0.93    281634
           1       0.43      0.71      0.54     30982

    accuracy                           0.88    312616
   macro avg       0.70      0.80      0.73    312616
weighted avg       0.91      0.88      0.89    312616

ROC–AUC: 0.9000398964443572
Confusion Matrix:
 [[252933  28701]
 [  9081  21901]]


In [ ]:
ensemble_prob = 0.8 * RF_prob + 0.2 * LR_prob
ensemble_pred = (ensemble_prob >= 0.5).astype(int)

print(classification_report(y_test, ensemble_pred))
print("ROC–AUC:", roc_auc_score(y_test, ensemble_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, ensemble_pred))

              precision    recall  f1-score   support

           0       0.97      0.90      0.93    281634
           1       0.44      0.71      0.54     30982

    accuracy                           0.88    312616
   macro avg       0.70      0.80      0.74    312616
weighted avg       0.91      0.88      0.89    312616

ROC–AUC: 0.9019209459986269
Confusion Matrix:
 [[253366  28268]
 [  9110  21872]]
